Developer acceptance on 2×T4. Includes deliberate test-only API failures; uses the exact published wheel and synthetic graphs.

In [ ]:
import contextlib,hashlib,importlib.metadata,json,os,subprocess,sys,time,urllib.request,zipfile,io,base64
from pathlib import Path
ROOT=Path('/kaggle/working');WORK=Path('/tmp')/('mgbfs-tuning-accept-'+str(os.getpid()));WORK.mkdir()
BASE='https://github.com/TryDotAtwo/MultiGPUBFS/releases/download/native-pip-20261017-v24/'
MANIFEST={'schema': 1, 'wheel': 'multigpubfs-0.2.0.dev24-py3-none-any.whl', 'sha256': '693917bbede1fe2f4808481f32335c3df0d4bda77e7f0eafe09e48537e36980d', 'size': 143420, 'source_commit': 'cf973ddf72d4b47dcc1b8d2fbcff9d9f75367259', 'native_index_sha256': '32564ff0a869dc1cf24d8255bc5d71acfdf6a3145de7497f757f4d28efec6ba5', 'native_components': 'native-components-20261017-v4', 'status': 'BUILT_NEEDS_INSTALLED_GPU_ACCEPTANCE'}
wheel=WORK/MANIFEST['wheel']
urllib.request.urlretrieve(BASE+MANIFEST['wheel'],wheel)
assert hashlib.sha256(wheel.read_bytes()).hexdigest()==MANIFEST['sha256']
subprocess.run([sys.executable,'-m','pip','install','--no-deps','--force-reinstall',str(wheel)],check=True)
subprocess.run([sys.executable,'-m','pip','install','cayleypy==0.2.0'],check=True)
assert importlib.metadata.version('multigpubfs')=='0.2.0.dev24'
os.environ['MGBFS_NATIVE_CACHE']=str(WORK/'cache');os.environ['MGBFS_PROFILE_CACHE']=str(WORK/'profiles')
import multigpubfs
multigpubfs.prefetch_native(sm=[75]);os.environ['MGBFS_NATIVE_OFFLINE']='1'
from multigpubfs.native_distribution import native_runtime
native,env=native_runtime();native_sha=hashlib.sha256(Path(native).read_bytes()).hexdigest()
manifest=json.loads((Path(native).parent.parent/'manifest.json').read_text())
assert manifest['native_primary_error_preserved'] is True
assert manifest['native_abort_failed_generic_comm'] is True
hardware=subprocess.check_output(['nvidia-smi','--query-gpu=name,compute_cap,driver_version','--format=csv,noheader'],text=True).strip().splitlines()
assert len(hardware)==2 and all(', 7.5,' in line for line in hardware),hardware
receipt={'schema':1,'status':'RUNNING','package':'0.2.0.dev24','wheel_sha256':MANIFEST['sha256'],'package_source_commit':MANIFEST['source_commit'],'native_binary_sha256':native_sha,'native_source_commit':manifest['native_binary_source_commit'],'hardware':hardware,'checks':[],
 'scope':'Physical two-rank T4 acceptance of healthy tuning, optional-probe fault containment and diagnostic retention; original 8-GPU CUDA 999 root cause not reproduced'}
def record(name,**data):
 receipt['checks'].append(dict(name=name,status='PASS',**data));(ROOT/'tuning-repair-acceptance.json').write_text(json.dumps(receipt,indent=2));print('TUNING_FIX_CHECK',name,flush=True)
started=time.monotonic()
try:
 from multigpubfs.deep_diagnostics import bfs_probe
 for fixture in ['permutation','repeated','matrix']:
  out=WORK/fixture;out.mkdir();record('exact-persistent-'+fixture,result=bfs_probe({'fixture':fixture,'devices':[0,1],'persistent':True},out))
 from multigpubfs import NativeOptions
 from cayleypy import CayleyGraph,PermutationGroups
 multigpubfs.enable_native(NativeOptions(devices=(0,1),max_seconds=60))
 with multigpubfs.persistent_graph_workers():
  for n in [3,4,5]:
   result=CayleyGraph(PermutationGroups.coxeter(n),device='cpu').bfs()
   assert result.diameter()==n*(n-1)//2
 record('cayleypy-api-persistent',degrees=[3,4,5],diameters=[3,6,10])
 child=WORK/'child.py';child.write_text("import contextlib,json,os,sys,warnings\nfrom pathlib import Path\nfrom multigpubfs.graph_definition import GraphDefinition\nfrom multigpubfs.native_distribution import native_runtime\nfrom multigpubfs.generic_session import persistent_graph_workers\nfrom multigpubfs.launch import run_graph\nmode,root,result_path=sys.argv[1:];root=Path(root);root.mkdir()\nnative,env=native_runtime();env=dict(env,MGBFS_TRACE_NATIVE_CALLS='1')\nif mode!='healthy':env['MGBFS_TEST_TUNING_FAULT']='main' if mode=='main-failure' else 'probe'\nidentity=[int(i==j) for i in range(4) for j in range(4)];generators=[]\nfor a in range(4):\n for b in range(a+1,4):\n  mx=identity.copy()\n  for c in range(4):mx[a*4+c],mx[b*4+c]=mx[b*4+c],mx[a*4+c]\n  generators.append((mx,257))\ng=GraphDefinition.matrix(4,4,generators,[256*v for v in identity])\noracle=g.exact_layers(24);layers=list(map(len,oracle))\ndef verify(report,out):\n assert report['status']=='COMPLETE' and report['layer_sizes']==layers,report\n states=json.loads((out/'states.json').read_text())\n assert sorted(states['current'])==oracle[-1]\n assert sorted(states['previous_small'])==oracle[-2]\n assert report['plan']['state_bytes']==8\npayload={'status':'PASS','mode':mode,'layers':layers,'states':24}\nwith persistent_graph_workers() if mode!='probe-oneshot' else contextlib.nullcontext() as pool:\n if mode=='main-failure':\n  try:run_graph(g,root/'main',devices=[0,1],autotune=False,max_seconds=30,_native_env=env)\n  except RuntimeError as e:\n   assert 'NATIVE_DISTRIBUTED_RANK_FAILED' in str(e)\n   evidence=root/'main/failure-evidence'\n  else:raise AssertionError('Main calculation failure was hidden')\n  phase=json.loads((evidence/'rank-1/phase.json').read_text());assert phase['native_error']['code']==999 and phase['native_last_error']['code']==(901 if phase['native_secondary_error_count'] else 999),phase\n  payload.update(main_error_propagated=True,first_fault=phase['native_error'],last_fault=phase['native_last_error'],no_main_restart=True)\n else:\n  with warnings.catch_warnings(record=True) as messages:\n   warnings.simplefilter('always')\n   report=run_graph(g,root/'first',devices=[0,1],autotune=True,max_seconds=180,_native_env=env)\n  verify(report,root/'first');profile=report['autotune']\n  if mode=='healthy':\n   assert 'tuning_failure' not in profile\n   again=run_graph(g,root/'second',devices=[0,1],autotune=True,max_seconds=180,_native_env=env);verify(again,root/'second')\n   payload.update(healthy_autotune=True,pids=report['rank_worker_pids'],second_pids=again['rank_worker_pids'])\n  else:\n   assert profile['status']=='TUNING_PROBE_FAILED_CONSERVATIVE_PROFILE' and profile['measured'] is False\n   failure=profile['tuning_failure'];evidence=Path(failure['evidence'])\n   assert evidence.is_dir() and str(evidence) in failure['error']\n   phase=json.loads((evidence/'rank-1/phase.json').read_text())\n   assert phase['native_error']['code']==999 and phase['native_error']['call']=='cudaMemcpyAsync' and phase['native_error']['layer']==2,phase\n   assert phase['native_last_error']['code']==(901 if phase['native_secondary_error_count'] else 999),phase\n   summary=json.loads((evidence/'summary.json').read_text());old={r['pid'] for c in summary['captures'] for r in c['ranks']}\n   assert old.isdisjoint(report['rank_worker_pids']),(old,report['rank_worker_pids'])\n   assert any('autotun' in str(w.message).lower() for w in messages)\n   context=json.loads((evidence.parents[1]/'failure-context.json').read_text())\n   assert context['error']==failure['error'] and str(evidence) in summary['error']\n   original=Path(context['original_error'].split('evidence=',1)[1]);assert not original.exists()\n   assert failure['source_graph_digest']==g.digest()\n   payload.update(optional_probe_contained=True,unmeasured_baseline=True,first_fault=phase['native_error'],last_fault=phase['native_last_error'],secondary_error_count=phase['native_secondary_error_count'],failed_pids=sorted(old),new_pids=report['rank_worker_pids'],durable_path_exists=True,probe_retried=False,main_restarted=False)\n   if pool is not None:\n    again=run_graph(g,root/'second',devices=[0,1],autotune=True,max_seconds=180,_native_env=env);verify(again,root/'second')\n    assert again['autotune']['status']==profile['status'] and again['rank_worker_pids']==report['rank_worker_pids']\n    payload['next_graph_reuses_healthy_workers']=True\nPath(result_path).write_text(json.dumps(payload,indent=2));print(json.dumps(payload),flush=True)\n")
 injector=WORK/'inject.so';c=WORK/'inject.c';c.write_text('#define _GNU_SOURCE\n#include <dlfcn.h>\n#include <stdio.h>\n#include <stdlib.h>\n#include <string.h>\nstatic int injected=0;\ntypedef int (*copy_fn)(void*,const void*,size_t,int,void*);\ntypedef int (*sync_fn)(void);\nstatic int eligible(void) {\n const char *mode=getenv("MGBFS_TEST_TUNING_FAULT"),*rank=getenv("RANK"),*path=getenv("MGBFS_JOB_PHASE_PATH");\n if(!mode||!rank||strcmp(rank,"1")||!path)return 0;\n FILE *f=fopen(path,"r");if(!f)return 0;char phase[16384];size_t n=fread(phase,1,sizeof(phase)-1,f);fclose(f);phase[n]=0;\n if(!strstr(phase,"\\"layer\\":2")||!strstr(phase,"\\"native_call_line\\":275"))return 0;\n int pilot=strstr(path,"/pilot-0/")!=NULL;\n const char *leaf=strrchr(path,\'/\');\n if(leaf&&!strncmp(leaf+1,"phase-",6)) {\n  char job[4096];snprintf(job,sizeof(job),"%.*s/job-%s",(int)(leaf-path),path,leaf+7);\n  f=fopen(job,"r");if(f){char text[65536];n=fread(text,1,sizeof(text)-1,f);fclose(f);text[n]=0;pilot=strstr(text,"/pilot-0/")!=NULL;}\n }\n return (!strcmp(mode,"probe")&&pilot)||(!strcmp(mode,"main")&&!pilot);\n}\nint cudaMemcpyAsync(void *dst,const void *src,size_t bytes,int kind,void *stream) {\n if(!injected&&bytes==4&&kind==2&&eligible()){injected=1;return 999;}\n copy_fn real=(copy_fn)dlsym(RTLD_NEXT,"cudaMemcpyAsync");return real(dst,src,bytes,kind,stream);\n}\nint cudaDeviceSynchronize(void) {\n sync_fn real=(sync_fn)dlsym(RTLD_NEXT,"cudaDeviceSynchronize");int result=real();\n return injected&&result==0?901:result;\n}\n')
 subprocess.run(['gcc','-shared','-fPIC','-O2','-o',str(injector),str(c),'-ldl'],check=True)
 for mode in ['healthy','probe-persistent','probe-oneshot','main-failure']:
  result=ROOT/(mode+'.json')
  run=subprocess.run([sys.executable,str(child),mode,str(WORK/mode),str(result)],env=dict(os.environ,LD_PRELOAD=str(injector)),capture_output=True,text=True,timeout=240)
  (ROOT/(mode+'.log')).write_text(run.stdout+'\n'+run.stderr)
  print(run.stdout[-4000:],run.stderr[-4000:],flush=True)
  if run.returncode:
   import shutil
   for evidence in (WORK/mode).glob('**/failure-evidence'):
    shutil.copytree(evidence,ROOT/('failed-'+mode+'-evidence'),dirs_exist_ok=True)
  assert run.returncode==0,(mode,run.returncode)
  data=json.loads(result.read_text());assert data['status']=='PASS';record(mode,**{k:v for k,v in data.items() if k!='status'})
 receipt.update(status='PASS',wall_seconds=time.monotonic()-started)
 (ROOT/'tuning-repair-acceptance.json').write_text(json.dumps(receipt,indent=2));print('TUNING_REPAIR_GPU_ACCEPTANCE_PASS',flush=True)
except BaseException as e:
 receipt.update(status='FAIL',error=str(e));(ROOT/'tuning-repair-acceptance.json').write_text(json.dumps(receipt,indent=2));raise
